# Predict labels with a fine‑tuned checkpoint (statistics or gender)

This notebook loads a fine‑tuned Transformer model from a saved checkpoint and predicts probabilities for the full unlabeled set and a set of conflicting descriptions. Use the TASK switch in the config cell to choose "stat" or "gender".

In [ ]:
# Configuration
# Choose task: "stat" for statistics, "gender" for gender classification
TASK = "stat"  # or "gender"

MODEL_ID = "xlm-roberta-large"  # must match the fine-tuned checkpoint base

# Base folders per task
if TASK == "stat":
    DATA_BASE = "./drive/MyDrive/Colab Notebooks/PRESS_2026"
    MODELS_BASE = "./drive/MyDrive/Colab Notebooks/PRESS_2026/models/stat"
    OUT_BASE = "./drive/MyDrive/Colab Notebooks/PRESS_2026"
    PATH_TO_MINE = f"{DATA_BASE}/stat_to_mine.feather"
    PATH_CONFLICTING = f"{DATA_BASE}/conflicting_descr_stat.feather"
    MODEL_CHECKPOINT = f"{MODELS_BASE}/large-checkpoint-2169"  # best checkpoint from training for stat
    OUT_UNLABELED_FEATHER = f"{OUT_BASE}/unlabeled_predicted.feather"
    OUT_CONFLICTING_FEATHER = f"{OUT_BASE}/conflicting_predicted_stat.feather"
    PROB_COL = "probability_is_statistics"
elif TASK == "gender":
    DATA_BASE = "./drive/MyDrive/Colab Notebooks/PRESS_2026"
    MODELS_BASE = "./drive/MyDrive/Colab Notebooks/PRESS_2026/models/gender"
    OUT_BASE = "./drive/MyDrive/Colab Notebooks/PRESS_2026"
    PATH_TO_MINE = f"{DATA_BASE}/gen_to_mine.feather"
    PATH_CONFLICTING = f"{DATA_BASE}/conflicting_descr_gen.feather"
    MODEL_CHECKPOINT = f"{MODELS_BASE}/large-checkpoint-3064"  # best checkpoint from training for gender
    OUT_UNLABELED_FEATHER = f"{OUT_BASE}/unlabeled_predicted_gen.feather"
    OUT_CONFLICTING_FEATHER = f"{OUT_BASE}/conflicting_predicted_gen.feather"
    PROB_COL = "probability_is_gender"
else:
    raise ValueError("TASK must be 'stat' or 'gender'")

# Inference parameters
BATCH_SIZE = 128  # adjust for GPU memory
MAX_TOKEN_LENGTH = None  # None => model max
SEED = 42

In [ ]:
# Setup: install (if needed) and import libs
# !pip install -U transformers datasets accelerate scikit-learn plotly pyarrow

import os
import random
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

In [ ]:
# Mount drive to import datasets
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Load data
# Choose label column based on TASK
LABEL_COL = "is_statistics" if TASK == "stat" else "is_gender"

set_to_mine = pd.read_feather(PATH_TO_MINE)
set_to_mine = set_to_mine.dropna(subset=["text_mining_description"])  # safety

unlabeled = set_to_mine[(set_to_mine[LABEL_COL] == False)].copy()
print(f"Unlabeled rows: {len(unlabeled):,}")

conflicting = pd.read_feather(PATH_CONFLICTING)
conflicting = conflicting.dropna(subset=["text_mining_description"]).copy()
print(f"Conflicting rows: {len(conflicting):,}")

In [ ]:
# Tokenizer and tokenization helpers
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        padding=True,               # dynamic padding
        truncation=True,
        max_length=MAX_TOKEN_LENGTH
    )

# Build HF datasets
unlabeled_texts = unlabeled["text_mining_description"].astype(str).tolist()
conflicting_texts = conflicting["text_mining_description"].astype(str).tolist()

unlabeled_ds = Dataset.from_dict({"text": unlabeled_texts}).map(tokenize_fn, batched=True)
conflicting_ds = Dataset.from_dict({"text": conflicting_texts}).map(tokenize_fn, batched=True)

In [ ]:
# Load fine‑tuned model from checkpoint for inference
model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT)

inference_args = TrainingArguments(
    output_dir="./tmp-results",  # no checkpointing for inference
    per_device_eval_batch_size=BATCH_SIZE,
    do_train=False,
    do_eval=False,
    fp16=torch.cuda.is_available(),
    report_to="none",
    logging_strategy="no",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=inference_args,
    processing_class=tokenizer, 
    data_collator=collator,
)

In [ ]:
# Predict unlabeled
pred_unlabeled = trainer.predict(unlabeled_ds)
probs_unlabeled = torch.softmax(torch.tensor(pred_unlabeled.predictions), dim=1)[:, 1].cpu().numpy()

unlabeled = unlabeled.copy()
unlabeled[PROB_COL] = probs_unlabeled

print("Unlabeled prediction done.")
unlabeled.to_feather(OUT_UNLABELED_FEATHER)
print(f"Saved: {OUT_UNLABELED_FEATHER}")

In [ ]:
# Predict conflicting descriptions
pred_conflicting = trainer.predict(conflicting_ds)
probs_conflicting = torch.softmax(torch.tensor(pred_conflicting.predictions), dim=1)[:, 1].cpu().numpy()

conflicting = conflicting.copy()
conflicting[PROB_COL] = probs_conflicting

print("Conflicting prediction done.")
conflicting.to_feather(OUT_CONFLICTING_FEATHER)
print(f"Saved: {OUT_CONFLICTING_FEATHER}")

In [ ]:
# Quick checks
print(unlabeled[[PROB_COL]].describe())
print(conflicting[[PROB_COL]].describe())

unlabeled.head(3), conflicting.head(3)